In [2]:
import xarray as xr
import numpy as np
import os
import cfgrib
    
DATA_PATH = "data/data.grib"
OUT_DIR = "data/processed"
LEVEL = 500 # pressure level (hPa)
INPUT_STEPS = 12 # past hours
OUTPUT_STEPS = 6 # future hours
TRAIN_RATIO = 0.6
VAL_RATIO = 0.2
TEST_RATIO = 0.2


os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
# Try to open GRIB file using cfgrib
try:
    ds = xr.open_dataset(
        DATA_PATH,
        engine="cfgrib",
        backend_kwargs={
            "indexpath": "",
            "filter_by_keys": {"typeOfLevel": "isobaricInhPa"}
        }
    )
    print("Successfully opened with cfgrib")
except Exception as e:
    print(f"cfgrib failed: {e}")
    print("\nTrying alternative: netcdf4 engine...")
    try:
        ds = xr.open_dataset(DATA_PATH, engine="netcdf4")
        print("Successfully opened with netcdf4")
    except Exception as e2:
        print(f"netcdf4 failed: {e2}")
        print("\nNote: GRIB file requires eccodes C library to read")
        print("Please ensure your data file exists and is in the correct format")
        raise

ds

Successfully opened with cfgrib


InvalidVersion: Invalid version: 'unknown'

InvalidVersion: Invalid version: 'unknown'

In [4]:
# Inspect and handle pressure level selection
current_level = float(ds.coords['isobaricInhPa'].values)
print(f"Current pressure level in dataset: {current_level} hPa")
print(f"Requested level: {LEVEL} hPa")

if 'isobaricInhPa' in ds.dims:
    # isobaricInhPa is a dimension, we can select directly
    u = ds['u'].sel(isobaricInhPa=LEVEL)
    v = ds['v'].sel(isobaricInhPa=LEVEL)
elif current_level == LEVEL:
    # Single-level dataset already at desired level
    u = ds['u']
    v = ds['v']
else:
    # Reopen dataset filtered to the desired pressure level
    print(f"Reopening dataset with level={LEVEL} hPa filter...")
    ds_level = xr.open_dataset(
        DATA_PATH,
        engine="cfgrib",
        backend_kwargs={
            "indexpath": "",
            "filter_by_keys": {
                "typeOfLevel": "isobaricInhPa",
                "level": LEVEL
            }
        }
    )
    u = ds_level['u']
    v = ds_level['v']

# Ensure latitude is ascending
u = u.sortby('latitude')
v = v.sortby('latitude')

print(u.shape)  # (T, H, W)

Current pressure level in dataset: 500.0 hPa
Requested level: 500 hPa
(2580, 141, 141)


In [5]:
u_np = u.values
v_np = v.values


wind = np.stack([u_np, v_np], axis=1)
print("Wind tensor:", wind.shape)

Wind tensor: (2580, 2, 141, 141)


In [6]:
T = wind.shape[0]


train_end = int(T * TRAIN_RATIO)
val_end = train_end + int(T * VAL_RATIO)


wind_train = wind[:train_end]
wind_val = wind[train_end:val_end]
wind_test = wind[val_end:]


print("Train steps:", wind_train.shape[0])
print("Val steps:", wind_val.shape[0])
print("Test steps:", wind_test.shape[0])

Train steps: 1548
Val steps: 516
Test steps: 516


In [7]:
mean = wind_train.mean(axis=(0, 2, 3), keepdims=True)
std = wind_train.std(axis=(0, 2, 3), keepdims=True) + 1e-6


wind_train_norm = (wind_train - mean) / std
wind_val_norm = (wind_val - mean) / std


print("Mean:", mean.flatten())
print("Std:", std.flatten())

Mean: [ 7.859272   -0.13663465]
Std: [11.886337   5.9452095]


In [8]:
np.save(os.path.join(OUT_DIR, "mean.npy"), mean)
np.save(os.path.join(OUT_DIR, "std.npy"), std)

In [9]:
#Sliding Window Function
def create_sliding_windows(data, input_steps, output_steps):
    X, Y = [], []
    T = data.shape[0]
    for t in range(T - input_steps - output_steps + 1):
        X.append(data[t:t+input_steps])
        Y.append(data[t+input_steps:t+input_steps+output_steps])
    return np.array(X), np.array(Y)

In [10]:
# Create sliding windows for training data
X_train, Y_train = create_sliding_windows(
wind_train_norm,
INPUT_STEPS,
OUTPUT_STEPS
)
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)

X_train: (1531, 12, 2, 141, 141)
Y_train: (1531, 6, 2, 141, 141)


In [11]:
# Create sliding windows for validation data
X_val, Y_val = create_sliding_windows(
wind_val_norm,
INPUT_STEPS,
OUTPUT_STEPS
)
print("X_val:", X_val.shape)
print("Y_val:", Y_val.shape)

X_val: (499, 12, 2, 141, 141)
Y_val: (499, 6, 2, 141, 141)


In [12]:
test_norm=(wind_test - mean) / std

In [13]:
# Create sliding windows for test data
X_test, Y_test = create_sliding_windows(
test_norm,
INPUT_STEPS,
OUTPUT_STEPS
)
print("X_test:", X_test.shape)
print("Y_test:", Y_test.shape)

X_test: (499, 12, 2, 141, 141)
Y_test: (499, 6, 2, 141, 141)


In [14]:
np.save(os.path.join(OUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUT_DIR, "Y_train.npy"), Y_train)
np.save(os.path.join(OUT_DIR, "X_val.npy"), X_val)
np.save(os.path.join(OUT_DIR, "Y_val.npy"), Y_val)
np.save(os.path.join(OUT_DIR, "X_test.npy"), X_test)
np.save(os.path.join(OUT_DIR, "Y_test.npy"), Y_test)